# Output Parser

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

## Output Parsing  
output parsing are classes that give model response in different formats  
output parser implements two main methods:  
- **Get instruction Formats**: This method returns a string containing instructions for how the output of a language model should be formatted.
- **Parser/Parse Method**: A method which takes string with instructions and returns different structure output(Json, Csv, Python Objects, Simple Text)

- Types of Output Parsing:  
    - StrOutputParser
    - JsonOutputParser
    - CSVOutputParser
    - StructuredOutputParser
      1. Pydantic
      2. Json

### 1. Pydantic Output Parser

In [18]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [19]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate, PromptTemplate)

base_url = "http://localhost:11434"
model = "llama3.2"

llm = ChatOllama(base_url = base_url, model = model)

In [20]:
class ProductReview(BaseModel):
    """Product review given by a user"""
    product_name: str = Field(description = "Name of the Product")
    review : str = Field(description = "User review text")
    rating : int = Field(description = "Rating from 1 to 5")
    verified_purchase : Optional[bool] = Field(description = "Whether the purchase is verified", default = None)

In [21]:
parser = PydanticOutputParser(pydantic_object = ProductReview)

In [25]:
instruction = parser.get_format_instructions()


In [26]:
print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Product review given by a user", "properties": {"product_name": {"description": "Name of the Product", "title": "Product Name", "type": "string"}, "review": {"description": "User review text", "title": "Review", "type": "string"}, "rating": {"description": "Rating from 1 to 5", "title": "Rating", "type": "integer"}, "verified_purchase": {"anyOf": [{"type": "boolean"}, {"type": "null"}], "default": null, "description": "Whether the purchase is verified", "title": "Verified Purchase"}}, "required": ["product_name", "review", "ra

In [29]:
prompt = PromptTemplate(
    template = '''
    write a Product Review based on the user query.
    Follow the formatting instructions carefully.
    
    {format_instruction}
    
    User Query : {query}
    
    Product Review:
    ''',
    input_variables = ['query'],
    partial_variables = {'format_instruction':parser.get_format_instructions()}
)

chain = prompt | llm 

In [30]:
output = chain.invoke({"query":"Write a review for apple airpods pro"})
print(output.content)

{"product_name": "Apple AirPods Pro", "review": "I recently purchased the Apple AirPods Pro and I must say, they have exceeded my expectations. The sound quality is amazing and the noise cancellation feature is incredibly effective. The design is sleek and compact, making them perfect for daily use.", "rating": 4, "verified_purchase": true}
